In [6]:
# from huggingface_hub import snapshot_download

# snapshot_download(repo_id="mesolitica/IMDA-TTS", repo_type="dataset", local_dir="./IMDA-TTS")

In [5]:
# !unzip IMDA-TTS/FEMALE_01.zip

In [7]:
!ls IMDA-TTS

FEMALE_01.zip  README.md  texts.json


In [8]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [9]:
with open('IMDA-TTS/texts.json') as fopen:
    d = json.load(fopen)
len(d)

6033

In [10]:
d[0]

{'filename': '0000.wav',
 'text': 'Author of the danger trail, Philip Steels, etc.'}

In [13]:
def loop(rows):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    rows, _ = rows

    data = []
    base = 'IMDA-TTS_audio'
    os.makedirs(base, exist_ok=True)

    for row in tqdm(rows):

        t = row['text'].strip()
        if len(t) < 2:
            continue

        f_new = os.path.join('FEMALE_01', row['filename'])
        audio_filename = f_new.replace('/', '_').replace('.wav', '.mp3')
        audio_filename = os.path.join(base, audio_filename)
        audio_np, sr = sf.read(f_new)
        if audio_np.ndim > 1:
            audio_np = audio_np.mean(axis=1)
        if audio_np.shape[0] < 10000:
            continue
        sf.write(audio_filename, audio_np, sr)
        
        data.append({
            'audio_filename': audio_filename,
            'text': t,
            'speaker': "IMDA-TTS"
        })
        
    return data

In [14]:
data = loop((d[:10], 0))

100%|██████████| 10/10 [00:00<00:00, 25.42it/s]


In [16]:
data = multiprocessing(d, loop, cores = 20)

100%|██████████| 301/301 [00:27<00:00, 11.12it/s]


In [17]:
len(data)

6033

In [18]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'IMDA-TTS_audio/FEMALE_01_0000.mp3',
 'text': 'Author of the danger trail, Philip Steels, etc.',
 'speaker': 'IMDA-TTS'}

In [19]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'IMDA-TTS')

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 350.61ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████|  342kB /  342kB, 1.71MB/s  
Processing Files (1 / 1): 100%|██████████|  342kB /  342kB,  856kB/s  
New Data Upload: 100%|██████████|  342kB /  342kB,  856kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.39 shards/s]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/36d370ef0551dc6dc6a794d39e6b4846198c3889', commit_message='Upload dataset', commit_description='', oid='36d370ef0551dc6dc6a794d39e6b4846198c3889', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [20]:
audio_files = [d['audio_filename'] for d in data]

with open('IMDA-TTS-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [23]:
folders = glob('IMDA-TTS_audio*')
folders = [f for f in folders if '.zip' not in f]
for f in folders:
    print(f)
    os.system(f'zip -rq {f}.zip {f}')

IMDA-TTS_audio
IMDA-TTS_audio_neucodec


In [24]:
from huggingface_hub import HfApi
api = HfApi()

for f in glob('IMDA-TTS_audio*.zip'):
    api.upload_file(
        path_or_fileobj=f,
        path_in_repo=f,
        repo_id="malaysia-ai/Multilingual-TTS",
        repo_type="dataset",
    )

Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  10%|█         | 29.9MB /  293MB,   ???B/s  
Processing Files (0 / 1):  56%|█████▋    |  165MB /  293MB,  676MB/s  
Processing Files (0 / 1):  99%|█████████▉|  290MB /  293MB,  651MB/s  
Processing Files (0 / 1):  99%|█████████▉|  291MB /  293MB,  435MB/s  
Processing Files (0 / 1): 100%|█████████▉|  292MB /  293MB,  262MB/s  
Processing Files (0 / 1): 100%|█████████▉|  292MB /  293MB,  219MB/s  
Processing Files (1 / 1): 100%|██████████|  293MB /  293MB,  164MB/s  
Processing Files (1 / 1): 100%|██████████|  293MB /  293MB,  146MB/s  
New Data Upload: 100%|██████████|  293MB /  293MB,  146MB/s  
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (1 / 1): 100%|██████████| 7.01MB / 7.01MB,   ???B/s  
Processing Files (1 / 1): 100%|██████████| 7.01MB / 7.01MB,  0.00B/s  
New Data Upload: 100%|██████████| 7.01MB / 7.01MB,  0.00B/s  
